In [ ]:
import sys
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider


/Library/Developer/CommandLineTools/usr/bin/python3


In [13]:
def print_code_finding(finding: CodeFinding) -> None:
    print(f"Title: {finding.title}")
    print(f"Severity: {finding.severity}")
    print()
    print("Explanation:")
    print(finding.explanation)
    print()
    print("Recommendation:")
    print(finding.recommendation)

In [2]:
model = OpenAIChatModel(
    model_name="qwen3:4b-instruct",
    provider=OllamaProvider(
        base_url="http://localhost:11434/v1"
    ),
)

---

## structural model

In [7]:
from typing import Literal


class CodeFinding(BaseModel):
    title: str
    severity: Literal["low", "medium", "high"]
    explanation: str
    recommendation: str

This is a normal Pydantic model.

It says that a `CodeFinding` must contain exactly these pieces of information:

- title
- severity
- explanation
- recommendation

In [8]:
agent = Agent(
    model,
    output_type=CodeFinding,
)

this is an angent that must return `CodeFinding`

previously we had:
`agent = Agent(model)` that return ordinary text.

now 
`output_type=CodeFinding` that means the final output must conform to the `CodeFinding` Pydantic model.

In [9]:
result = await agent.run(
    """
    Analyse this code:

    user_age = input("Age: ")
    print("You are " + user_age)

    Identify one software engineering problem.
    Return a structured finding.
    """
)

print(result.output)

title='Missing Input Validation' severity='high' explanation="The code does not validate whether the input provided by the user is a valid integer or numeric value. If the user enters a non-numeric value (e.g., 'abc'), the program will raise a ValueError when attempting to convert the input to an integer. This could lead to runtime errors and incorrect behavior." recommendation='Add input validation to ensure that the user input is a valid numeric value before processing. For example, use a try-except block to catch ValueError exceptions and prompt the user to re-enter a valid age if the input is invalid.'


In [10]:
print(type(result.output))

<class '__main__.CodeFinding'>


In [14]:
print_code_finding(result.output)

Title: Missing Input Validation
Severity: high

Explanation:
The code does not validate whether the input provided by the user is a valid integer or numeric value. If the user enters a non-numeric value (e.g., 'abc'), the program will raise a ValueError when attempting to convert the input to an integer. This could lead to runtime errors and incorrect behavior.

Recommendation:
Add input validation to ensure that the user input is a valid numeric value before processing. For example, use a try-except block to catch ValueError exceptions and prompt the user to re-enter a valid age if the input is invalid.
